# 06 · Event-Probability (Binary) Forecasts

The forecast is a single number $p_t=\Pr(\text{event}_t\mid\mathcal I_{t-1})\in[0,1]$, e.g.
"70% chance of rain", "35% chance that monthly sales grow by more than 5% year-on-year", "12% chance of a stock-out".

## Theory

### Two ways to get $p_t$
1. **Derive from a predictive distribution**: $p_t=\Pr(Y_t>c)=1-\hat F_t(c)$. A single model then gives a coherent
   probability for *any* threshold $c$ (the **exceedance curve**).
2. **Direct probabilistic classification**: logistic regression, gradient boosting with log-loss, etc., trained on
   the 0/1 outcome. Simple, but you need a separate model per threshold.

### Scoring
* **Brier score** $\text{BS}=\frac1N\sum(p_t-o_t)^2$ (strictly proper, bounded)
* **Log loss** $-\frac1N\sum[o\log p+(1-o)\log(1-p)]$ (strictly proper, punishes confident mistakes heavily)
* **Brier skill score** $\text{BSS}=1-\text{BS}/\text{BS}_{\text{climatology}}$, where climatology always forecasts the base rate
* **Murphy decomposition**: $\text{BS}=\underbrace{\text{REL}}_{\text{calibration, ↓}}-\underbrace{\text{RES}}_{\text{resolution, ↑}}+\underbrace{\text{UNC}}_{\bar o(1-\bar o)}$
  * Reliability: do 30% forecasts verify 30% of the time?
  * Resolution: do the forecasts *separate* event from non-event situations? (sharpness that is also correct)
* **Reliability diagram** (calibration curve) + **sharpness histogram** (how often each $p$ is issued)
* **ROC / AUC** measure *discrimination only*. They ignore calibration and are **not** proper scores.

### Recalibration
A forecaster with good resolution but poor reliability can be fixed after the fact by learning a monotone map
$p\mapsto g(p)$: **isotonic regression** (non-parametric) or **Platt / logistic** scaling. Fit it on held-out data only.

### From probability to decision: the cost–loss model
Protecting costs $C$; an unprotected event costs $L$. Expected expense is minimised by acting when $p > C/L$.
Probabilistic forecasts let *each user* pick their own threshold. A single yes/no forecast cannot.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
from scipy.optimize import isotonic_regression
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(6)

## 1. Setup: four stores, one-step-ahead, event = "YoY growth > 5%"
Parameters are estimated on the first 60 months of each store (2015–2019). One-step forecasts are then made for
2020–2024 with the Kalman filter (no look-ahead): 4 stores × 60 months = 240 probability forecasts.

In [2]:
retail = pf.make_retail_hierarchy()
CUT, GROWTH = 60, np.log(1.05)
frames = []
for s in pf.BOTTOM_NAMES:
    ly = np.log(retail[s])
    fit = SARIMAX(ly.iloc[:CUT], order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).fit(disp=False)
    pr = SARIMAX(ly, order=(0, 1, 1), seasonal_order=(0, 1, 1, 12)).filter(fit.params).get_prediction(start=CUT)
    thr = (ly.shift(12) + GROWTH).iloc[CUT:]
    df = pd.DataFrame({"store": s, "mu": np.asarray(pr.predicted_mean), "sd": np.sqrt(np.asarray(pr.var_pred_mean)),
                       "thr": thr.to_numpy(), "logy": ly.iloc[CUT:].to_numpy()}, index=ly.index[CUT:])
    # lagged YoY growth features for the direct classifier (known at forecast time)
    g = ly - ly.shift(12)
    df["g1"], df["g2"] = g.shift(1).iloc[CUT:].to_numpy(), g.shift(2).iloc[CUT:].to_numpy()
    df["base_rate"] = np.mean((ly - ly.shift(12)).iloc[12:CUT] > GROWTH)
    frames.append(df)
ev = pd.concat(frames)
ev["o"] = (ev.logy > ev.thr).astype(int)
print(f"{len(ev)} forecasts, event frequency = {ev.o.mean():.2f}")

240 forecasts, event frequency = 0.41


## 2. Four forecasters
* **Model**: $p=1-\Phi\big((c-\mu)/\sigma\big)$ from the SARIMA predictive distribution
* **Over-confident model**: same $\mu$ but $\sigma/3$ (a typical failure: ignoring parameter uncertainty, etc.)
* **Logistic**: direct classifier on the last two observed YoY growth rates, trained on 2016–2019
* **Climatology**: each store's historical base rate (calibrated, zero resolution)

In [3]:
ev["p_model"] = stats.norm.sf(ev.thr, ev.mu, ev.sd)
ev["p_overconf"] = stats.norm.sf(ev.thr, ev.mu, ev.sd / 3)
ev["p_clim"] = ev.base_rate

# direct classifier trained on the pre-2020 period
train_rows = []
for s in pf.BOTTOM_NAMES:
    ly = np.log(retail[s]); g = ly - ly.shift(12)
    d = pd.DataFrame({"g1": g.shift(1), "g2": g.shift(2), "o": (g > GROWTH).astype(int)}).iloc[14:CUT]
    train_rows.append(d)
tr = pd.concat(train_rows)
logit = sm.Logit(tr.o, sm.add_constant(tr[["g1", "g2"]])).fit(disp=0)
ev["p_logit"] = np.asarray(logit.predict(sm.add_constant(ev[["g1", "g2"]])))
print(logit.params.round(2).to_dict())

FC = {"Model (SARIMA)": "p_model", "Over-confident": "p_overconf", "Logistic": "p_logit", "Climatology": "p_clim"}

def auc(o, p):                     # Mann–Whitney formulation
    r = stats.rankdata(p); n1 = o.sum(); n0 = len(o) - n1
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

bs_clim = pf.brier_score(ev.o, ev.p_clim)
tab = {}
for name, col in FC.items():
    dec = pf.brier_decomposition(ev.o, ev[col], n_bins=10)
    tab[name] = {"Brier": dec["brier"], "BSS vs clim": 1 - dec["brier"] / bs_clim, "log loss": pf.log_loss(ev.o, ev[col]),
                 "reliability ↓": dec["reliability"], "resolution ↑": dec["resolution"], "uncertainty": dec["uncertainty"],
                 "AUC": auc(ev.o.to_numpy(), ev[col].to_numpy())}
pd.DataFrame(tab).T

{'const': -0.44, 'g1': 10.36, 'g2': -4.37}


,Brier,BSS vs clim,log loss,reliability ↓,resolution ↑,uncertainty,AUC
Model (SARIMA),0.179,0.277,0.552,0.012,7.271e-02,0.242,0.804
Over-confident,0.201,0.191,1.189,0.029,7.061e-02,0.242,0.804
Logistic,0.234,0.055,0.661,0.016,2.345e-02,0.242,0.654
Climatology,0.248,0.000,0.689,0.007,4.340e-04,0.242,0.502


In [4]:
FC_COLOR = dict(zip(FC, pf.SERIES))
fig = pf.subplots(1, 2, titles=["Reliability diagram", "Sharpness: how often each probability was issued"], height=440)
for k, (name, col) in enumerate(FC.items()):
    if name != "Climatology":
        pf.add_reliability(fig, ev.o, ev[col], n_bins=8, name=name, color=FC_COLOR[name], row=1, col=1, diagonal=k == 0)
    pf.add_step_hist(fig, ev[col], bins=10, range_=(0, 1), name=name, color=FC_COLOR[name], row=1, col=2,
                     density=False, showlegend=name == "Climatology")
fig.update_xaxes(title_text="forecast probability", row=1, col=2)
fig.update_yaxes(title_text="number of forecasts", row=1, col=2)
fig.show()

* The **over-confident** forecaster has the *same AUC* as the model (identical ranking of cases), but a worse Brier score
  and a much worse log loss (worse even than climatology!). Its reliability curve is too flat: when it says 90%, the
  event happens far less often. AUC is blind to calibration.
* **Climatology** has zero resolution. Beating it (BSS > 0) is the minimum bar.

## 3. Recalibration with isotonic regression
Fit the monotone map on 2020–2022 (calibration) and apply it to 2023–2024 (test).

In [5]:
cal_mask = ev.index < "2023-01-01"
def fit_isotonic(p, o):
    order = np.argsort(p)
    xs, ys = np.asarray(p)[order], isotonic_regression(np.asarray(o, float)[order]).x
    return lambda q: np.interp(q, xs, ys)

rows = {}
for name in ["Over-confident", "Model (SARIMA)"]:
    col = FC[name]
    g = fit_isotonic(ev.loc[cal_mask, col], ev.loc[cal_mask, "o"])
    p_test, o_test = ev.loc[~cal_mask, col], ev.loc[~cal_mask, "o"]
    p_rec = np.clip(g(p_test), 0.01, 0.99)
    rows[f"{name}: raw"] = {"Brier": pf.brier_score(o_test, p_test), "log loss": pf.log_loss(o_test, p_test)}
    rows[f"{name}: isotonic"] = {"Brier": pf.brier_score(o_test, p_rec), "log loss": pf.log_loss(o_test, p_rec)}
pd.DataFrame(rows).T

,Brier,log loss
Over-confident: raw,0.210,1.651
Over-confident: isotonic,0.198,0.673
Model (SARIMA): raw,0.191,0.611
Model (SARIMA): isotonic,0.198,0.673


Recalibration helps the over-confident forecaster substantially. For an already-calibrated model, it adds little and
can even hurt slightly with small calibration sets (isotonic maps are step functions that over-fit).

## 4. One model, any threshold: the exceedance curve
A distributional forecast answers "P(growth > c)" for **every** c at once. A classifier needs one model per threshold.

In [6]:
row = ev[ev.store == "North_A"].iloc[-1]
growth = np.linspace(-0.10, 0.20, 200)
last_year = row.thr - GROWTH
p_exceed = stats.norm.sf(last_year + np.log1p(growth), row.mu, row.sd)
fig = go.Figure(go.Scatter(x=growth * 100, y=p_exceed, mode="lines", name="P(growth > c)", line=dict(color=pf.SERIES[0], width=2),
                           hovertemplate="P(growth > %{x:.1f}%) = %{y:.2f}<extra></extra>"))
pf.add_vline(fig, 5, 0, 1, "5% event threshold", color=pf.INK["muted"], dash="dash")
pf.add_vline(fig, (np.exp(row.logy - last_year) - 1) * 100, 0, 1, "actual growth", color=pf.ACTUAL, dash="solid", width=2)
fig.update_layout(title=f"Exceedance curve, North_A {row.name:%b %Y}: one forecast answers every threshold", height=380,
                  xaxis_title="YoY growth threshold c (%)", yaxis_title="P(growth > c)")
fig.show()

## 5. Decisions: the cost–loss model
Suppose an extra shipment costs $C$ and running out costs $L$ when demand surges. The optimal rule is to ship
if $p>C/L$. **Relative economic value** $V=\frac{E_{\text{clim}}-E_{\text{fcst}}}{E_{\text{clim}}-E_{\text{perfect}}}$
(1 = perfect information, 0 = no better than climatology).

In [7]:
o = ev.o.to_numpy(); obar = o.mean()
ratios = np.linspace(0.05, 0.95, 19)
fig = go.Figure()
for name, col in FC.items():
    p = ev[col].to_numpy(); V = []
    for r in ratios:                                  # use L = 1, C = r
        act = p > r
        E_f = np.mean(act * r + (~act) * o * 1.0)
        E_clim = min(r, obar)                         # always act or never act, whichever is cheaper
        E_perf = obar * r
        V.append((E_clim - E_f) / (E_clim - E_perf))
    fig.add_trace(go.Scatter(x=ratios, y=V, mode="lines+markers", name=name, line=dict(color=pf.SERIES[list(FC).index(name)], width=2),
                             marker=dict(size=8), hovertemplate=name + ": V = %{y:.2f}<extra></extra>"))
fig.add_hline(y=0, line=dict(color=pf.INK["axis"], width=1))
fig.update_layout(title="Who benefits from the forecast? (value curve; 0 = no better than climatology)", height=400,
                  hovermode="x unified", yaxis_range=[-0.5, 1], xaxis_title="cost / loss ratio C/L", yaxis_title="relative value V")
fig.show()

Calibrated probabilistic forecasts give positive value over a wide range of users (C/L ratios). The over-confident
forecaster loses value for users with extreme cost ratios, because those users take its extreme probabilities at face value.

### Exercises
1. Change the event to "sales fall by more than 3%". How does the base rate change the Brier score's scale?
2. Platt-scale the over-confident forecaster (logistic regression on logit(p)) and compare it with isotonic regression.
3. Make 3-month-ahead event forecasts with `get_forecast(3)` at each origin. How do resolution and value decay?